In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import numpy as np
import pandas as pd
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from imblearn.over_sampling import RandomOverSampler
from sklearn.preprocessing import LabelEncoder
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# 读取数据
train_df = pd.read_csv('new_cicddos2019_dataset.csv')


# 移除无关列
train_df.drop(columns=['Unnamed: 0', 'Class'], inplace=True)


# 检查是否有缺失值
print("Train missing values:", train_df.isnull().sum().sum())


# One-hot 编码列
# cols = ['proto', 'state', 'service']
# 
# def one_hot(df, cols):
#     for col in cols:
#         dummies = pd.get_dummies(df[col], prefix=col, drop_first=False)
#         df = pd.concat([df, dummies], axis=1)
#         df.drop(col, axis=1, inplace=True)
#     return df

# 合并数据进行统一处理
combined_data = train_df

# 分离目标变量
target = combined_data.pop('Label')

# One-hot 编码
# combined_data = one_hot(combined_data, cols)

# Min-Max 归一化
def normalize(df):
    result = df.copy()
    for col in df.columns:
        max_val = df[col].max()
        min_val = df[col].min()
        if max_val > min_val:
            result[col] = (df[col] - min_val) / (max_val - min_val)
    return result

# 归一化数据
normalized_data = normalize(combined_data)

# 添加目标列
normalized_data['Class'] = target

# 检查是否有缺失值
assert not normalized_data.isnull().values.any(), "归一化后的数据存在缺失值"

# 分离特征和标签
X = normalized_data.drop(columns=['Class']).values
y = normalized_data['Class'].values
print(X.shape)
# 将目标变量转换为整数编码
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

Train missing values: 0
(203880, 77)


In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

class CNN_BiLSTM(nn.Module):
    def __init__(self):
        super(CNN_BiLSTM, self).__init__()

        # 1D 卷积层
        self.conv1 = nn.Conv1d(in_channels=1, out_channels=64, kernel_size=64, padding='same')
        self.relu = nn.ReLU()
        self.pool1 = nn.MaxPool1d(kernel_size=10)
        self.batchnorm1 = nn.BatchNorm1d(64)

        # 第一个 BiLSTM
        self.bilstm1 = nn.LSTM(input_size=64, hidden_size=64, batch_first=True, bidirectional=True)

        # 第二次池化 + 归一化
        self.pool2 = nn.MaxPool1d(kernel_size=5)
        self.batchnorm2 = nn.BatchNorm1d(128)

        # 第二个 BiLSTM
        self.bilstm2 = nn.LSTM(input_size=128, hidden_size=128, batch_first=True, bidirectional=True)

        # Dropout 层
        self.dropout = nn.Dropout(0.6)

        # 全连接层
        self.fc = nn.Linear(128 * 2, 18)  # 双向 LSTM 使得 hidden_size 变成 128*2
        self.softmax = nn.Softmax(dim=1)

    def forward(self, x):
        # Conv1D 期望输入形状: (batch, in_channels, seq_len)
        x = self.conv1(x)
        x = self.relu(x)
        x = self.pool1(x)
        x = self.batchnorm1(x)

        # **调整形状以适应 LSTM**
        x = x.permute(0, 2, 1)  # (batch, seq_len, channels)

        # BiLSTM 1
        x, _ = self.bilstm1(x)

        # 交换维度用于 BatchNorm 和 MaxPooling
        x = x.permute(0, 2, 1)  # (batch, feature_dim, seq_len)
        x = self.pool2(x)
        x = self.batchnorm2(x)

        # **调整形状以适应第二个 LSTM**
        x = x.permute(0, 2, 1)  # (batch, seq_len, feature_dim)
        x, _ = self.bilstm2(x)

        # 取最后一个时间步的输出
        x = x[:, -1, :]

        # Dropout + 全连接 + Softmax
        x = self.dropout(x)
        x = self.fc(x)
        x = self.softmax(x)

        return x



In [3]:
from collections import Counter
from sklearn import metrics
from tqdm import tqdm
from sklearn.preprocessing import LabelEncoder
# 设置超参数
batch_size = 32
epochs =10    
learning_rate = 0.001
k=10
# 交叉验证
kfold = StratifiedKFold(n_splits=k, shuffle=True, random_state=40)
# oversample = RandomOverSampler(sampling_strategy='minority')


# 统计结果
oos_pred = []
# 初始化结果列表
oos_accuracies = []
last_fold_y_true = []
last_fold_y_pred = []

# 初始化模型
model = CNN_BiLSTM().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

# 遍历折叠
# 过采样少数类
#X_resampled, y_resampled = oversample.fit_resample(X, y_encoded)

for fold, (train_idx, test_idx) in enumerate(kfold.split(X, y_encoded), start=1):
#for fold, (train_idx, test_idx) in enumerate(kfold.split(X_resampled, y_resampled), start=1):
    # 划分数据集
    X_train, X_test = X[train_idx], X[test_idx]
    y_train, y_test = y_encoded[train_idx], y_encoded[test_idx]

    #X_train, X_test = X_resampled[train_idx], X_resampled[test_idx]
    #y_train, y_test = y_resampled[train_idx], y_resampled[test_idx]

    # 过采样少数类
    # X_train, y_train = oversample.fit_resample(X_train, y_train)

    # 转换为 PyTorch TensorDataset
    train_dataset = TensorDataset(torch.tensor(X_train, dtype=torch.float32),
                                  torch.tensor(y_train, dtype=torch.long))
    test_dataset = TensorDataset(torch.tensor(X_test, dtype=torch.float32),
                                torch.tensor(y_test, dtype=torch.long))

    # DataLoader
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

    print(f"Fold {fold}: {len(train_loader.dataset)} train samples, {len(test_loader.dataset)} validation samples")

    for epoch in range(epochs):
        model.train()
        epoch_loss = 0.0
        correct = 0
        total = 0

        # 使用 tqdm 显示进度条
        progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}")
        for batch_data, batch_labels in progress_bar:
            batch_data, batch_labels = batch_data.to(device), batch_labels.to(device)

            batch_data = batch_data.unsqueeze(1)  # 添加通道维度
            optimizer.zero_grad()
            outputs = model(batch_data)
            loss = criterion(outputs, batch_labels)
            loss.backward()
            optimizer.step()

            # 累积损失和准确性
            epoch_loss += loss.item()
            _, preds = torch.max(outputs, 1)
            correct += (preds == batch_labels).sum().item()
            total += batch_labels.size(0)

            # 更新进度条
            progress_bar.set_postfix({"loss": f"{loss.item():.4f}"})

        # 计算每轮的平均损失和准确率
        epoch_loss /= len(train_loader)
        epoch_acc = correct / total
        print(f"Epoch {epoch+1}/{epochs} - Loss: {epoch_loss:.4f}, Accuracy: {epoch_acc:.4f}")    

    model.eval()
    y_true, y_pred = [], []
    with torch.no_grad():
        for batch_data, batch_labels in test_loader:
            batch_data, batch_labels = batch_data.to(device), batch_labels.to(device)
            batch_data = batch_data.unsqueeze(1)  # 添加通道维度
            outputs = model(batch_data)
            _, preds = torch.max(outputs, 1)

            y_true.extend(batch_labels.cpu().numpy())
            y_pred.extend(preds.cpu().numpy())

    # 测试集各类别数量
    # test_class_counts = Counter(y_true)
    # print("\nTest Set Class Distribution:")
    # for label, count in sorted(test_class_counts.items()):
    #     print(f"  Class {label}: {count}")

    # 计算准确率
    acc = metrics.accuracy_score(y_true, y_pred)
    oos_accuracies.append(acc)
    print(f"Fold {fold} Accuracy: {acc:.4f}")

    # 保存最后一折的结果
    if fold == kfold.get_n_splits():
        last_fold_y_true = y_true
        last_fold_y_pred = y_pred

# 保存模型
model_save_path = "CIC-CNN-BiLSTM.pth"
torch.save(model, model_save_path)
print(f"Complete model saved to {model_save_path}")

# 输出每一折的准确率
print("Fold Accuracies:")
for i, acc in enumerate(oos_accuracies, start=1):
    print(f"  Fold {i}: {acc:.4f}")
 

Fold 1: 183492 train samples, 20388 validation samples


Epoch 1/10:   0%|          | 0/5735 [00:00<?, ?it/s]E:\shendu\anaconda3\envs\pytorch1\lib\site-packages\torch\nn\modules\conv.py:306: UserWarning: Using padding='same' with even kernel lengths and odd dilation may require a zero-padded copy of the input be created (Triggered internally at C:\cb\pytorch_1000000000000\work\aten\src\ATen\native\Convolution.cpp:1041.)
  return F.conv1d(input, weight, bias, self.stride,
Epoch 1/10: 100%|██████████| 5735/5735 [00:29<00:00, 193.51it/s, loss=1.9828]


Epoch 1/10 - Loss: 2.1832, Accuracy: 0.8014


Epoch 2/10: 100%|██████████| 5735/5735 [00:27<00:00, 207.37it/s, loss=2.2315]


Epoch 2/10 - Loss: 2.1627, Accuracy: 0.8190


...

Epoch 8/10 - Loss: 2.1542, Accuracy: 0.8274


Epoch 9/10: 100%|██████████| 5735/5735 [00:28<00:00, 200.92it/s, loss=2.4852]


Epoch 9/10 - Loss: 2.1540, Accuracy: 0.8276


Epoch 10/10: 100%|██████████| 5735/5735 [00:28<00:00, 197.76it/s, loss=1.9815]


Epoch 10/10 - Loss: 2.1541, Accuracy: 0.8274
Fold 10 Accuracy: 0.8273
Complete model saved to CIC-CNN-BiLSTM.pth
Fold Accuracies:
  Fold 1: 0.8277
  Fold 2: 0.8289
  Fold 3: 0.8242
  Fold 4: 0.8275
  Fold 5: 0.8231
  Fold 6: 0.8267
  Fold 7: 0.8277
  Fold 8: 0.8285
  Fold 9: 0.8289
  Fold 10: 0.8273


In [4]:
from sklearn.metrics import confusion_matrix, classification_report
import numpy as np

# 分类类别
categories = ['1', '2', '3', '4', '5', '6','7', '8', '9', '10','11', '12', '13', '14', '15', '16','17', '18']

# 混淆矩阵（最后一折的结果）
last_cm = confusion_matrix(last_fold_y_true, last_fold_y_pred, labels=range(18))

print("\nLast Fold Confusion Matrix:")
print(last_cm)

report = classification_report(last_fold_y_true, last_fold_y_pred, target_names=categories)
print("\nClassification Report:")
print(report)

# 从混淆矩阵提取所有类别的统计信息
total_samples = last_cm.sum()  # 总样本数
correct_predictions = np.trace(last_cm)  # 对角线元素的和，即正确分类的总数

# 总体准确率（直接计算）
overall_accuracy = correct_predictions / total_samples

# 初始化总体指标
overall_TP = 0
overall_FN = 0
overall_FP = 0
overall_TN = 0


# 重新计算分类报告中的 TP、FP、FN、TN
detection_rates = {}
false_positive_rates = {}

for i, category in enumerate(categories):
    TP = last_cm[i, i]
    FN = last_cm[i, :].sum() - TP
    FP = last_cm[:, i].sum() - TP
    TN = total_samples - (TP + FP + FN)

    if category != "1":  # 统计攻击类型的总 TP 和 FN
        overall_TP += TP
        overall_FN += FN
    else:  # 统计正常类型的 TN 和 FP
        overall_TN += TN
        overall_FP += FP

    # 每类检测率和误报率
    detection_rates[category] = TP / (TP + FN) if (TP + FN) > 0 else 0.0
    false_positive_rates[category] = FP / (FP + TN) if (FP + TN) > 0 else 0.0

    print(f"Category: {category}")
    print(f"  Detection Rate (DR): {detection_rates[category]:.4f}")
    print(f"  False Positive Rate (FPR): {false_positive_rates[category]:.4f}")

# 总体检测率（攻击类型的 DR）和误报率（Normal 的 FPR）
overall_dr = overall_TP / (overall_TP + overall_FN) if (overall_TP + overall_FN) > 0 else 0.0
overall_fpr = overall_FP / (overall_FP + overall_TN) if (overall_FP + overall_TN) > 0 else 0.0

print("\nOverall Metrics:")
print(f"  Accuracy (Acc): {overall_accuracy:.4f}")
print(f"  Detection Rate (DR): {overall_dr:.4f}")
print(f"  False Positive Rate (FPR): {overall_fpr:.4f}")


Last Fold Confusion Matrix:
[[3709   12    0    0    0    0    0    0    0    1    0    0    3    0
     0    2    0    0]
 [   4  146    0    0    4    0  120    0    0   82    2    0    0    0
     9    0    0    0]
 [   0   51    0    0    0    0   86    0    0    6    0    0    1    0
     0    0    0    0]
 [   0    2    0    0    0    0   32    0    0  583    0    0    0    0
     4    1    0    0]
 [  23    1    0    0 4592    0    0    0    0    6    0    0    0    0
     1    0    0    0]
 [   2   15    0    0    0    0    0    0    0   10   29    0    0    2
     1    0    0    0]
 [   0   29    0    0    0    0  230    0    0    7    6    0    0    0
     0    0    0    0]
 [   0    5    0    0    7    0    0   13    0   23    0    0    0    0
   994    0    0    0]
 [   0   30    0    0    0    0  152    0    0    9    0    0    0    0
     0    0    0    0]
 [   0    0    0    0    0    0   37    0    0  808    0    0    0    1
     7    0    0    0]
 [   2   19    0    0

E:\shendu\anaconda3\envs\pytorch1\lib\site-packages\sklearn\metrics\_classification.py:1469: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
E:\shendu\anaconda3\envs\pytorch1\lib\site-packages\sklearn\metrics\_classification.py:1469: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
E:\shendu\anaconda3\envs\pytorch1\lib\site-packages\sklearn\metrics\_classification.py:1469: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
